# Project 3 — Australian Income by Postcode
## Exploratory Data Analysis

**Data sources:**
- ATO Taxation Statistics 2022–23, Table 6B — individuals by postcode | [data.gov.au](https://data.gov.au/data/dataset/taxation-statistics-postcode-data)
- ABS SEIFA 2021 — Postal Area, Indexes (socio-economic advantage/disadvantage by postcode) | [abs.gov.au](https://www.abs.gov.au/statistics/people/people-and-communities/socio-economic-indexes-areas-seifa-australia/2021)
- Australian Postcodes dataset — suburb names + remoteness classification | [github.com/matthewproctor](https://github.com/matthewproctor/australianpostcodes)

**Tools:** Python (pandas, matplotlib, seaborn)

**Goal:** Understand the structure, distribution, and patterns in individual income across Australian postcodes. Enrich with SEIFA and suburb data before building the Tableau dashboard.

## 1. Setup & Data Load

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (12, 5)

# --- ATO Income Data ---
df_raw = pd.read_excel(
    'ato_table6b.xlsx',
    sheet_name='Table 6B',
    header=1
)
print(f'ATO raw shape: {df_raw.shape}')

# --- Suburb + Remoteness Lookup ---
# Source: https://github.com/matthewproctor/australianpostcodes
df_suburbs_raw = pd.read_csv('australian_postcodes.csv')
print(f'Suburb lookup raw shape: {df_suburbs_raw.shape}')

# --- SEIFA 2021 (Postal Area) ---
# Source: ABS - https://www.abs.gov.au/statistics/people/people-and-communities/socio-economic-indexes-areas-seifa-australia/2021
# File: seifa_poa_2021.xlsx
df_seifa_raw = pd.read_excel('seifa_poa_2021.xlsx', sheet_name=None)
print(f'SEIFA sheets: {list(df_seifa_raw.keys())}')

## 2. Clean ATO Income Data

In [ ]:
# Keep only actual postcode rows
df = df_raw[pd.to_numeric(df_raw['Postcode'], errors='coerce').notna()].copy()
df['Postcode'] = df['Postcode'].astype(int)

rename_map = {
    'State/ Territory1':                                   'state',
    'Statistical Area Level 4 (SA4)2':                    'sa4',
    'Postcode':                                            'postcode',
    'Individuals\nno.':                                   'individuals',
    'Taxable income or loss4\n$':                         'taxable_income',
    'Net tax\n$':                                         'net_tax',
    'Salary or wages\n$':                                 'salary',
    'Gross interest\n$':                                  'gross_interest',
    'Dividends franked\n$':                               'dividends_franked',
    'Capital gains net capital gain\n$':                  'capital_gains',
    'Gross rent\n$':                                      'gross_rent',
    'Net rent4\n$':                                       'net_rent',
    'HELP debt balance\n$':                               'help_debt',
    'Australian government allowances and payments\n$':   'govt_allowances',
    'Net income or loss from business\n$':                'business_income',
    'People with private health insurance\nno.':          'private_health_no',
}

df = df[list(rename_map.keys())].rename(columns=rename_map)

num_cols = [c for c in df.columns if c not in ['state', 'sa4', 'postcode']]
for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df = df[df['individuals'] > 0].copy()
print(f'ATO clean shape: {df.shape} | Postcodes: {df["postcode"].nunique()}')
df.head(3)

## 3. Prepare Suburb & Remoteness Lookup

In [ ]:
# Multiple suburbs per postcode — take the entry WITH remoteness data first,
# then by lowest id as tiebreaker
suburb_lookup = (
    df_suburbs_raw
    .assign(has_remoteness=df_suburbs_raw['RA_2021_NAME'].notna().astype(int))
    .sort_values(['postcode', 'has_remoteness', 'id'], ascending=[True, False, True])
    .groupby('postcode')
    .first()
    .reset_index()
    [['postcode', 'locality', 'RA_2021_NAME']]
    .rename(columns={
        'locality':    'suburb',
        'RA_2021_NAME': 'remoteness'
    })
)
suburb_lookup['suburb'] = suburb_lookup['suburb'].str.title()

print(f'Suburb lookup: {len(suburb_lookup)} postcodes')
print(f'Remoteness categories: {suburb_lookup["remoteness"].dropna().unique()}')
print(f'Missing remoteness: {suburb_lookup["remoteness"].isna().sum()} postcodes (unusual/PO box postcodes)')
suburb_lookup.head(5)

## 4. Prepare SEIFA Data

In [ ]:
# Load Table 1 from SEIFA file
# Sheet: Table 1 | Header spans rows 5-6; data starts row 7
# Using header=5 (0-indexed row 6) and skipping the merged label rows above
seifa = pd.read_excel(
    'seifa_poa_2021.xlsx',
    sheet_name='Table 1',
    header=5  # row 6 in Excel = index 5 in pandas
)

print('Shape:', seifa.shape)
print('Columns:', seifa.columns.tolist()[:12])
seifa.head(3)

In [ ]:
# Extract postcode, IRSD decile (col 2) and IRSAD decile (col 4)
# Also keeping IRSAD score (col 3) for the scatter plot
# Column positions from Table 1:
#   col 0 = POA Code
#   col 1 = IRSD Score
#   col 2 = IRSD Decile
#   col 3 = IRSAD Score
#   col 4 = IRSAD Decile

seifa_clean = seifa.iloc[:, [0, 2, 3, 4]].copy()
seifa_clean.columns = ['postcode', 'seifa_irsd_decile', 'seifa_irsad_score', 'seifa_irsad_decile']

seifa_clean['postcode']           = pd.to_numeric(seifa_clean['postcode'],           errors='coerce')
seifa_clean['seifa_irsd_decile']  = pd.to_numeric(seifa_clean['seifa_irsd_decile'],  errors='coerce')
seifa_clean['seifa_irsad_score']  = pd.to_numeric(seifa_clean['seifa_irsad_score'],  errors='coerce')
seifa_clean['seifa_irsad_decile'] = pd.to_numeric(seifa_clean['seifa_irsad_decile'], errors='coerce')
seifa_clean = seifa_clean.dropna(subset=['postcode'])
seifa_clean['postcode'] = seifa_clean['postcode'].astype(int)

print(f'SEIFA clean: {len(seifa_clean)} postcodes')
print(f'IRSAD decile range: {seifa_clean["seifa_irsad_decile"].min():.0f} - {seifa_clean["seifa_irsad_decile"].max():.0f}')
print(f'IRSD decile range:  {seifa_clean["seifa_irsd_decile"].min():.0f} - {seifa_clean["seifa_irsd_decile"].max():.0f}')
seifa_clean.head(5)

## 5. Join All Three Datasets
Left join from ATO data — we keep all ATO postcodes as the base, and enrich with suburb names, remoteness, and SEIFA scores where available.

In [ ]:
df = df.merge(suburb_lookup, on='postcode', how='left')
df = df.merge(seifa_clean,   on='postcode', how='left')

print(f'Final shape: {df.shape}')
print(f'Suburb match rate:     {df["suburb"].notna().mean():.1%}')
print(f'SEIFA match rate:      {df["seifa_irsad_decile"].notna().mean():.1%}')
print(f'Remoteness match rate: {df["remoteness"].notna().mean():.1%}')
df[['postcode','suburb','state','remoteness','seifa_irsad_decile','seifa_irsd_decile']].head(5)

## 6. Calculate Per-Person Averages
Raw dollar totals reflect postcode population size, not income level. Dividing by number of individual lodgers gives a comparable per-person figure across postcodes.

In [ ]:
n = df['individuals']

df['avg_taxable_income']  = df['taxable_income']    / n
df['avg_net_tax']         = df['net_tax']           / n
df['avg_salary']          = df['salary']            / n
df['avg_gross_interest']  = df['gross_interest']    / n
df['avg_dividends']       = df['dividends_franked'] / n
df['avg_capital_gains']   = df['capital_gains']     / n
df['avg_gross_rent']      = df['gross_rent']        / n
df['avg_net_rent']        = df['net_rent']          / n
df['avg_help_debt']       = df['help_debt']         / n
df['avg_govt_allowances'] = df['govt_allowances']   / n
df['avg_business_income'] = df['business_income']   / n
df['pct_private_health']  = (df['private_health_no'] / n * 100).round(1)
df['effective_tax_rate']  = (df['net_tax'] / df['taxable_income'].replace(0, np.nan) * 100).round(2)

print('Per-person metrics calculated.')
df[['postcode','suburb','avg_taxable_income','effective_tax_rate','seifa_irsad_score','remoteness']].head(8)

## 7. Dataset Overview

In [ ]:
avg_cols = ['avg_taxable_income','avg_salary','avg_gross_interest','avg_dividends',
            'avg_capital_gains','avg_gross_rent','avg_help_debt',
            'avg_govt_allowances','avg_business_income','pct_private_health',
            'effective_tax_rate','seifa_irsad_score']

summary = df[avg_cols].describe().T[['mean','50%','min','max','std']]
summary.columns = ['Mean','Median','Min','Max','Std Dev']

def fmt(val, col):
    if col in ['pct_private_health','effective_tax_rate']:
        return f'{val:.1f}%'
    if col == 'seifa_irsad_score':
        return f'{val:,.0f}'
    return f'${val:,.0f}'

for c in summary.columns:
    summary[c] = [fmt(summary.loc[i, c], i) for i in summary.index]

print(f'Dataset: {len(df):,} postcodes | {df["individuals"].sum():,.0f} total lodgers')
summary

## 8. Income Distribution Across Postcodes
Is income roughly evenly distributed, or are a handful of wealthy postcodes pulling the average up?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
data = df['avg_taxable_income'].dropna()

for ax, yscale, title in [
    (axes[0], 'linear', 'Distribution of Avg Taxable Income by Postcode'),
    (axes[1], 'log',    'Same — Log Scale (reveals outlier tail)'),
]:
    ax.hist(data, bins=60, color='steelblue', edgecolor='white', linewidth=0.4)
    ax.axvline(data.mean(),   color='tomato', linestyle='--', linewidth=1.5, label=f'Mean: ${data.mean():,.0f}')
    ax.axvline(data.median(), color='gold',   linestyle='--', linewidth=1.5, label=f'Median: ${data.median():,.0f}')
    ax.set_yscale(yscale)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}k'))
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('Avg Taxable Income per Lodger')
    ax.set_ylabel('Number of Postcodes' + (' (log)' if yscale == 'log' else ''))
    ax.legend()

plt.tight_layout()
plt.savefig('plot_income_distribution.png', bbox_inches='tight')
plt.show()
print(f'Skewness: {data.skew():.2f} — positive skew confirms a long right tail of high-income postcodes')

## 9. Top & Bottom 10 Postcodes by Average Income

## 10. Average Income by State

In [ ]:
state_income = (
    df.groupby('state')
    .apply(lambda g: g['taxable_income'].sum() / g['individuals'].sum())
    .reset_index()
    .rename(columns={0: 'weighted_avg_income'})
    .sort_values('weighted_avg_income', ascending=True)
)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(state_income['state'], state_income['weighted_avg_income'], color='steelblue')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}k'))
ax.set_title('Weighted Average Taxable Income by State/Territory (2022-23)', fontsize=12)
ax.set_xlabel('Average Taxable Income per Lodger')
for bar, val in zip(bars, state_income['weighted_avg_income']):
    ax.text(bar.get_width() + 200, bar.get_y() + bar.get_height()/2,
            f'${val:,.0f}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('plot_income_by_state.png', bbox_inches='tight')
plt.show()

## 11. SEIFA vs Income — Does Socio-Economic Advantage Predict Income?
SEIFA IRSAD scores rank areas by relative advantage and disadvantage using Census data. This scatter plot tests whether the ATO income data aligns with — or diverges from — the SEIFA ranking.

In [ ]:
seifa_plot = df.dropna(subset=['seifa_irsad_score','avg_taxable_income']).copy()

# Cap income at 95th percentile for readability
cap = seifa_plot['avg_taxable_income'].quantile(0.95)
seifa_plot_capped = seifa_plot[seifa_plot['avg_taxable_income'] <= cap]

fig, ax = plt.subplots(figsize=(12, 6))
scatter = ax.scatter(
    seifa_plot_capped['seifa_irsad_score'],
    seifa_plot_capped['avg_taxable_income'],
    alpha=0.4, s=15, c=seifa_plot_capped['seifa_irsad_decile'],
    cmap='RdYlGn'
)

# Trend line
z = np.polyfit(seifa_plot_capped['seifa_irsad_score'], seifa_plot_capped['avg_taxable_income'], 1)
p = np.poly1d(z)
x_line = np.linspace(seifa_plot_capped['seifa_irsad_score'].min(), seifa_plot_capped['seifa_irsad_score'].max(), 100)
ax.plot(x_line, p(x_line), color='steelblue', linewidth=2, label='Trend line')

corr = seifa_plot_capped[['seifa_irsad_score','avg_taxable_income']].corr().iloc[0,1]
ax.set_title(f'SEIFA IRSAD Score vs Average Taxable Income by Postcode (r = {corr:.2f})', fontsize=12)
ax.set_xlabel('SEIFA IRSAD Score (higher = more advantaged)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}k'))
ax.set_ylabel('Average Taxable Income per Lodger')
ax.legend()
plt.colorbar(scatter, ax=ax, label='IRSAD Decile (1=most disadvantaged, 10=most advantaged)')
plt.tight_layout()
plt.savefig('plot_seifa_vs_income.png', bbox_inches='tight')
plt.show()
print(f'Pearson r = {corr:.3f}')
print('Note: 5% of highest-income outlier postcodes excluded from scatter for readability')

## 12. Correlation Matrix
Which income indicators move together geographically? Strong correlations reveal the underlying structure of wealth and disadvantage across postcodes.

In [ ]:
corr_cols = {
    'avg_taxable_income':  'Avg Income',
    'avg_salary':          'Salary',
    'avg_gross_interest':  'Interest',
    'avg_dividends':       'Dividends',
    'avg_capital_gains':   'Capital Gains',
    'avg_gross_rent':      'Gross Rent',
    'avg_help_debt':       'HELP Debt',
    'avg_govt_allowances': 'Govt Allowances',
    'avg_business_income': 'Business Income',
    'pct_private_health':  'Private Health %',
    'effective_tax_rate':  'Eff. Tax Rate',
    'seifa_irsad_score':   'SEIFA IRSAD Score',
}

corr_df = df[list(corr_cols.keys())].rename(columns=corr_cols)
corr_matrix = corr_df.corr()

fig, ax = plt.subplots(figsize=(13, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f',
    cmap='RdYlGn', center=0, vmin=-1, vmax=1,
    linewidths=0.5, ax=ax, annot_kws={'size': 9}
)
ax.set_title('Correlation Matrix — Per-Person Income Indicators + SEIFA by Postcode', fontsize=13, pad=15)
plt.tight_layout()
plt.savefig('plot_correlation_matrix.png', bbox_inches='tight')
plt.show()

print('Correlations with Avg Income (strongest first):')
print(corr_matrix['Avg Income'].drop('Avg Income').sort_values(ascending=False).to_string())

## 13. Outlier Detection
Identifying postcodes that sit far outside the normal income range. Critical for Tableau — outliers compress the colour scale and make the map unreadable without correction.

In [ ]:
q1  = df['avg_taxable_income'].quantile(0.25)
q3  = df['avg_taxable_income'].quantile(0.75)
iqr = q3 - q1
upper_fence = q3 + 1.5 * iqr

outliers = df[df['avg_taxable_income'] > upper_fence].sort_values('avg_taxable_income', ascending=False)

print(f'IQR upper fence: ${upper_fence:,.0f}')
print(f'Outlier postcodes: {len(outliers)}')
print(f'Tableau recommendation: clamp colour scale max at ${upper_fence:,.0f}')
print()
print(outliers[['postcode','suburb','state','avg_taxable_income','seifa_irsad_decile','individuals']].to_string(index=False))

fig, ax = plt.subplots(figsize=(12, 4))
ax.boxplot(df['avg_taxable_income'].dropna(), vert=False, patch_artist=True,
           boxprops=dict(facecolor='steelblue', alpha=0.6),
           medianprops=dict(color='gold', linewidth=2),
           flierprops=dict(marker='o', color='tomato', markersize=4, alpha=0.5))

top = outliers.iloc[0]
label = top['suburb'] if pd.notna(top['suburb']) else str(int(top['postcode']))
ax.annotate(
    f"{label} ({top['state']})\n${top['avg_taxable_income']:,.0f}",
    xy=(top['avg_taxable_income'], 1), xytext=(top['avg_taxable_income'] * 0.72, 1.32),
    arrowprops=dict(arrowstyle='->', color='tomato'),
    fontsize=9, color='tomato'
)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}k'))
ax.set_title('Boxplot — Average Taxable Income by Postcode (outliers in red)', fontsize=12)
ax.set_xlabel('Average Taxable Income per Lodger')
ax.set_yticks([])
plt.tight_layout()
plt.savefig('plot_outliers_boxplot.png', bbox_inches='tight')
plt.show()

## 14. Income Composition — High vs Low Income Postcodes
Where does income actually come from in wealthy vs lower-income postcodes? This reveals whether high earners are salary-dependent or investment-driven.

In [ ]:
top_q    = df[df['avg_taxable_income'] >= df['avg_taxable_income'].quantile(0.75)]
bottom_q = df[df['avg_taxable_income'] <= df['avg_taxable_income'].quantile(0.25)]

def income_composition(group):
    n = group['individuals'].sum()
    return {
        'Salary':          group['salary'].sum() / n,
        'Business Income': group['business_income'].clip(lower=0).sum() / n,
        'Capital Gains':   group['capital_gains'].sum() / n,
        'Dividends':       group['dividends_franked'].sum() / n,
        'Gross Rent':      group['gross_rent'].sum() / n,
        'Govt Allowances': group['govt_allowances'].sum() / n,
        'Interest':        group['gross_interest'].sum() / n,
    }

comp_df = pd.DataFrame({
    'Top Quartile':    income_composition(top_q),
    'Bottom Quartile': income_composition(bottom_q),
})

fig, ax = plt.subplots(figsize=(12, 6))
comp_df.T.plot(kind='bar', ax=ax, colormap='Set2', edgecolor='white')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}k'))
ax.set_title('Income Composition — Top vs Bottom Income Quartile Postcodes', fontsize=12)
ax.set_xlabel('')
ax.set_ylabel('Average $ per Lodger')
ax.set_xticklabels(['Top Quartile Postcodes', 'Bottom Quartile Postcodes'], rotation=0)
ax.legend(title='Income Source', bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()
plt.savefig('plot_income_composition.png', bbox_inches='tight')
plt.show()
print(comp_df.map(lambda x: f'${x:,.0f}').to_string())

## 15. Income Quintile Distribution by State
Dividing postcodes into five equal income bands and comparing how each state distributes across them reveals geographic inequality that a simple average obscures.

In [ ]:
df['income_quintile'] = pd.qcut(df['avg_taxable_income'], q=5,
                                  labels=['Q1 Bottom','Q2','Q3','Q4','Q5 Top'])

quintile_state = df.groupby(['state','income_quintile']).size().unstack(fill_value=0)
quintile_state_pct = quintile_state.div(quintile_state.sum(axis=1), axis=0).mul(100).round(1)

fig, ax = plt.subplots(figsize=(13, 6))
quintile_state_pct.plot(kind='bar', ax=ax, colormap='RdYlGn', edgecolor='white', width=0.8)
ax.set_title('Income Quintile Distribution by State/Territory', fontsize=13)
ax.set_xlabel('')
ax.set_ylabel('% of Postcodes')
ax.set_xticklabels(quintile_state_pct.index, rotation=0)
ax.legend(title='Income Quintile', bbox_to_anchor=(1.01, 1), loc='upper left')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}%'))
plt.tight_layout()
plt.savefig('plot_quintile_by_state.png', bbox_inches='tight')
plt.show()

print('% of postcodes in TOP quintile by state:')
print(quintile_state_pct['Q5 Top'].sort_values(ascending=False).to_string())
print()
print('% of postcodes in BOTTOM quintile by state:')
print(quintile_state_pct['Q1 Bottom'].sort_values(ascending=False).to_string())

## 16. HELP Debt Concentration by Region
HELP debt is a proxy for graduate concentration — where university-educated workers live and whether their earnings are sufficient to service their debt. Analysis restricted to postcodes with 500+ lodgers for robustness.

In [ ]:
df_large = df[df['individuals'] >= 500].copy()

# Top postcodes by HELP debt
top_help = df_large.nlargest(15, 'avg_help_debt')[['postcode','suburb','state','sa4','avg_help_debt','avg_salary','avg_taxable_income','individuals']]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left — top 15 postcodes
ax1 = axes[0]
label = top_help['suburb'].fillna(top_help['postcode'].astype(str)) + ' (' + top_help['state'] + ')'
bars = ax1.barh(label, top_help['avg_help_debt'], color='steelblue')
ax1.invert_yaxis()
ax1.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}k'))
ax1.set_title('Top 15 Postcodes — Avg HELP Debt per Lodger', fontsize=11)
ax1.set_xlabel('Avg HELP Debt')
for bar, val in zip(bars, top_help['avg_help_debt']):
    ax1.text(bar.get_width() * 0.02, bar.get_y() + bar.get_height()/2,
             f'${val:,.0f}', va='center', fontsize=8, color='white', fontweight='bold')

# Right — avg HELP debt by state
ax2 = axes[1]
state_help = df.groupby('state').apply(lambda g: g['help_debt'].sum() / g['individuals'].sum()).sort_values(ascending=True)
bars2 = ax2.barh(state_help.index, state_help.values, color='steelblue')
ax2.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.1f}k'))
ax2.set_title('Avg HELP Debt per Lodger by State', fontsize=11)
ax2.set_xlabel('Avg HELP Debt')
for bar, val in zip(bars2, state_help.values):
    ax2.text(bar.get_width() + 10, bar.get_y() + bar.get_height()/2,
             f'${val:,.0f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('plot_help_debt.png', bbox_inches='tight')
plt.show()

print('Top 10 SA4 regions by avg HELP debt:')
sa4_help = df.groupby('sa4').apply(lambda g: g['help_debt'].sum() / g['individuals'].sum()).nlargest(10)
print(sa4_help.round(0).to_string())

## 17. Effective Tax Rate vs Theoretical Rate
Comparing the actual effective tax rate against the theoretical rate based on ATO 2022-23 tax brackets. The gap reveals the aggregate effect of deductions, offsets, and tax minimisation strategies at the postcode level — highly relevant to ATO compliance work.

In [ ]:
def theoretical_tax_rate(income):
    if pd.isna(income) or income <= 0:
        return np.nan
    if income <= 18200:
        return 0.0
    elif income <= 45000:
        return (income - 18200) * 0.19 / income * 100
    elif income <= 120000:
        return (5092 + (income - 45000) * 0.325) / income * 100
    elif income <= 180000:
        return (29467 + (income - 120000) * 0.37) / income * 100
    else:
        return (51667 + (income - 180000) * 0.45) / income * 100

df['theoretical_etr'] = df['avg_taxable_income'].apply(theoretical_tax_rate)
df['etr_gap'] = df['theoretical_etr'] - df['effective_tax_rate']  # negative = paying more than theoretical

# By income quintile
etr_q = df.groupby('income_quintile')[['avg_taxable_income','effective_tax_rate','theoretical_etr','etr_gap']].mean().round(2)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left — actual vs theoretical ETR by quintile
ax1 = axes[0]
x = range(len(etr_q))
width = 0.35
ax1.bar([i - width/2 for i in x], etr_q['effective_tax_rate'], width, label='Actual ETR', color='steelblue')
ax1.bar([i + width/2 for i in x], etr_q['theoretical_etr'], width, label='Theoretical ETR', color='lightcoral', alpha=0.8)
ax1.set_xticks(x)
ax1.set_xticklabels(etr_q.index, rotation=15)
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda y, _: f'{y:.0f}%'))
ax1.set_title('Actual vs Theoretical ETR by Income Quintile', fontsize=11)
ax1.set_ylabel('Effective Tax Rate')
ax1.legend()

# Right — ETR gap distribution across all postcodes
ax2 = axes[1]
ax2.hist(df['etr_gap'].dropna(), bins=50, color='steelblue', edgecolor='white', linewidth=0.4)
ax2.axvline(0, color='tomato', linestyle='--', linewidth=1.5, label='Zero gap (actual = theoretical)')
ax2.axvline(df['etr_gap'].mean(), color='gold', linestyle='--', linewidth=1.5, label=f'Mean gap: {df["etr_gap"].mean():.1f}%')
ax2.set_title('Distribution of ETR Gap Across All Postcodes', fontsize=11)
ax2.set_xlabel('ETR Gap (theoretical minus actual) — negative = paying more than theoretical')
ax2.set_ylabel('Number of Postcodes')
ax2.legend()

plt.tight_layout()
plt.savefig('plot_etr_gap.png', bbox_inches='tight')
plt.show()

print('ETR analysis by income quintile:')
print(etr_q.to_string())
print()
print('Negative etr_gap = paying MORE than theoretical (LITO/low income offset effect at bottom quintile)')
print('Note: taxable income here is already post-deduction — theoretical rate applied to post-deduction income')

## 18. Postcode Segmentation — K-Means Clustering
Using k-means clustering to group postcodes into segments based on multiple income and socio-economic variables simultaneously. This goes beyond simple income ranking to reveal meaningfully different postcode types — useful as a filter dimension in the Tableau dashboard.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

df['investment_income_pct'] = ((df['dividends_franked'] + df['capital_gains'] + df['gross_rent'] + df['gross_interest'])
                                / df['taxable_income'].replace(0, np.nan) * 100)
df['welfare_dependency_pct'] = (df['govt_allowances'] / df['taxable_income'].replace(0, np.nan) * 100)

cluster_features = [
    'avg_taxable_income', 'avg_salary', 'avg_dividends',
    'avg_capital_gains', 'avg_help_debt', 'avg_govt_allowances',
    'pct_private_health', 'investment_income_pct', 'welfare_dependency_pct'
]

cluster_df = df[cluster_features].dropna()
valid_idx = cluster_df.index
scaler = StandardScaler()
X_scaled = scaler.fit_transform(cluster_df)
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
df.loc[valid_idx, 'cluster_id'] = kmeans.fit_predict(X_scaled)
df['cluster_id'] = df['cluster_id'].astype('Int64')

# Label clusters based on profiles
cluster_labels = {
    0: 'Comfortable Suburbs',
    1: 'Struggling Areas',
    2: 'Wealthy Investor Class',
    3: 'Mainstream Australia',
    4: 'Extreme Outlier'
}
df['cluster_label'] = df['cluster_id'].map(cluster_labels)

# Profile table
profile = df.groupby('cluster_label').agg(
    n_postcodes=('postcode','count'),
    avg_income=('avg_taxable_income','mean'),
    avg_salary=('avg_salary','mean'),
    avg_dividends=('avg_dividends','mean'),
    avg_capital_gains=('avg_capital_gains','mean'),
    avg_help_debt=('avg_help_debt','mean'),
    avg_govt_allowances=('avg_govt_allowances','mean'),
    pct_private_health=('pct_private_health','mean'),
    investment_pct=('investment_income_pct','mean'),
    welfare_pct=('welfare_dependency_pct','mean'),
).round(0)

# Rename columns short for clean display
display_income = profile[['n_postcodes','avg_income','avg_salary','avg_dividends','avg_capital_gains']].copy()
display_income.columns = ['N','Avg Income','Salary','Dividends','Cap Gains']
display_social = profile[['n_postcodes','avg_help_debt','avg_govt_allowances','pct_private_health','investment_pct','welfare_pct']].copy()
display_social.columns = ['N','HELP Debt','Govt Allow','Pvt Health%','Invest%','Welfare%']
print('CLUSTER PROFILES — INCOME')
print(display_income.to_string())
print()
print('CLUSTER PROFILES — SOCIAL INDICATORS')
print(display_social.to_string())

# Visualise cluster profiles
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left — avg income by cluster
ax1 = axes[0]
cluster_income = df[df['cluster_label'] != 'Extreme Outlier'].groupby('cluster_label')['avg_taxable_income'].mean().sort_values()
colours = ['#d73027','#fc8d59','#fee08b','#91cf60','#1a9850']
bars = ax1.barh(cluster_income.index, cluster_income.values, color=colours)
ax1.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}k'))
ax1.set_title('Average Income by Postcode Segment', fontsize=11)
ax1.set_xlabel('Avg Taxable Income per Lodger')
for bar, val in zip(bars, cluster_income.values):
    ax1.text(bar.get_width() * 0.02, bar.get_y() + bar.get_height()/2,
             f'${val:,.0f}', va='center', fontsize=9, color='white', fontweight='bold')

# Right — income composition by cluster
ax2 = axes[1]
comp_clusters = df[df['cluster_label'] != 'Extreme Outlier'].groupby('cluster_label').apply(
    lambda g: pd.Series({
        'Salary':      g['avg_salary'].mean(),
        'Investment':  (g['avg_dividends'] + g['avg_capital_gains'] + g['avg_gross_rent']).mean(),
        'Govt Allow':  g['avg_govt_allowances'].mean(),
    })
)
comp_clusters.plot(kind='bar', ax=ax2, colormap='Set2', edgecolor='white')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}k'))
ax2.set_title('Income Composition by Postcode Segment', fontsize=11)
ax2.set_xlabel('')
ax2.set_ylabel('Avg $ per Lodger')
ax2.set_xticklabels(ax2.get_xticklabels(), rotation=20, ha='right')
ax2.legend(title='Income Source')

plt.tight_layout()
plt.savefig('plot_clusters.png', bbox_inches='tight')
plt.show()

# State breakdown by cluster
print('State distribution by cluster:')
print(df[df['cluster_label'] != 'Extreme Outlier'].groupby(['cluster_label','state']).size().unstack(fill_value=0).to_string())

## 19. Key EDA Findings

In [ ]:
top1    = df.nlargest(1, 'avg_taxable_income').iloc[0]
bottom1 = df.nsmallest(1, 'avg_taxable_income').iloc[0]
nat_avg = df['taxable_income'].sum() / df['individuals'].sum()
nat_med = df['avg_taxable_income'].median()
skew    = df['avg_taxable_income'].skew()
seifa_r = df[['seifa_irsad_score','avg_taxable_income']].corr().iloc[0,1]

top_suburb    = top1['suburb']    if pd.notna(top1['suburb'])    else str(int(top1['postcode']))
bottom_suburb = bottom1['suburb'] if pd.notna(bottom1['suburb']) else str(int(bottom1['postcode']))

# Quintile stats
act_top_pct = df[df['state']=='ACT']['income_quintile'].value_counts(normalize=True).get('Q5 Top', 0) * 100
tas_bot_pct = df[df['state']=='TAS']['income_quintile'].value_counts(normalize=True).get('Q1 Bottom', 0) * 100
wa_bot_pct  = df[df['state']=='WA']['income_quintile'].value_counts(normalize=True).get('Q1 Bottom', 0) * 100

# HELP debt stats
vic_help = df[df['state']=='VIC']['help_debt'].sum() / df[df['state']=='VIC']['individuals'].sum()
nt_help  = df[df['state']=='NT']['help_debt'].sum()  / df[df['state']=='NT']['individuals'].sum()

# ETR stats
etr_gap_mean = df['etr_gap'].mean()
q1_gap = df[df['income_quintile']=='Q1 Bottom']['etr_gap'].mean()
q5_gap = df[df['income_quintile']=='Q5 Top']['etr_gap'].mean()

# Cluster counts
cluster_counts = df['cluster_label'].value_counts()

print('=' * 65)
print('KEY EDA FINDINGS — Project 3: Australian Income by Postcode')
print('=' * 65)
print(f'Data sources: ATO 2022-23 | ABS SEIFA 2021 | Postcode lookup')
print(f'Postcodes analysed:          {len(df):,}')
print(f'Total individual lodgers:    {df["individuals"].sum():,.0f}')
print()
print('--- INCOME DISTRIBUTION ---')
print(f'National avg taxable income: ${nat_avg:,.0f}')
print(f'National median (postcode):  ${nat_med:,.0f}')
print(f'Skewness: {skew:.2f} — right-skewed; wealthy postcodes pull mean above median')
print(f'Highest income: {top_suburb} {int(top1["postcode"])} ({top1["state"]}) — ${top1["avg_taxable_income"]:,.0f}')
print(f'Lowest income:  {bottom_suburb} {int(bottom1["postcode"])} ({bottom1["state"]}) — ${bottom1["avg_taxable_income"]:,.0f}')
print(f'Range: {top1["avg_taxable_income"]/bottom1["avg_taxable_income"]:.0f}x difference top to bottom')
print(f'Outlier postcodes (IQR):     {len(df[df["avg_taxable_income"] > df["avg_taxable_income"].quantile(0.75) + 1.5*(df["avg_taxable_income"].quantile(0.75)-df["avg_taxable_income"].quantile(0.25))]):}')
print()
print('--- GEOGRAPHIC INEQUALITY (QUINTILE BY STATE) ---')
print(f'ACT: {act_top_pct:.0f}% of postcodes in top income quintile — most advantaged state')
print(f'TAS: {tas_bot_pct:.0f}% of postcodes in bottom quintile — most disadvantaged state')
print(f'WA:  {wa_bot_pct:.0f}% in bottom quintile — likely mining income effect')
print()
print('--- SEIFA vs INCOME ---')
print(f'Correlation: r = {seifa_r:.3f} — strong positive but not perfect')
print(f'  → Socio-economic advantage predicts income but notable exceptions exist')
print()
print('--- INCOME COMPOSITION ---')
print('Top quartile: income driven by capital gains, dividends and rent — not just salary')
print('Bottom quartile: more reliant on govt allowances, far less investment income')
print()
print('--- HELP DEBT CONCENTRATION ---')
print(f'VIC leads all states: ${vic_help:,.0f} avg HELP debt per lodger')
print(f'NT lowest: ${nt_help:,.0f} avg HELP debt per lodger')
print('Melbourne Inner dominates top 15 postcodes by HELP debt — graduate concentration')
print()
print('--- EFFECTIVE TAX RATE GAP ---')
print(f'All quintiles pay MORE effective tax than theoretical rate on taxable income')
print(f'Bottom quintile gap: {q1_gap:.2f}% (largest gap — LITO phase-out effect)')
print(f'Top quintile gap: {q5_gap:.2f}%')
print('Note: taxable income is already post-deduction per ATO definition')
print()
print('--- POSTCODE SEGMENTATION (K-MEANS, 5 CLUSTERS) ---')
for label, count in cluster_counts.items():
    if label != 'Extreme Outlier':
        avg_inc = df[df['cluster_label']==label]['avg_taxable_income'].mean()
        print(f'{label}: {count} postcodes | avg income ${avg_inc:,.0f}')


## 20. Export Clean CSV for Tableau

In [ ]:
export_cols = [
    'postcode', 'suburb', 'state', 'sa4', 'remoteness',
    'individuals', 'taxable_income', 'net_tax',
    'avg_taxable_income', 'avg_net_tax', 'effective_tax_rate',
    'avg_salary', 'avg_gross_interest', 'avg_dividends',
    'avg_capital_gains', 'avg_gross_rent', 'avg_net_rent',
    'avg_help_debt', 'avg_govt_allowances', 'avg_business_income',
    'pct_private_health', 'seifa_irsad_score', 'seifa_irsad_decile', 'seifa_irsd_decile',
    'income_quintile', 'investment_income_pct', 'welfare_dependency_pct',
    'theoretical_etr', 'etr_gap', 'cluster_id', 'cluster_label'
]

# Keep only columns that exist in df
export_cols = [c for c in export_cols if c in df.columns]
df_export = df[export_cols].round(2)
df_export.to_csv('postcodes_cleaned.csv', index=False)

print(f'Exported {len(df_export):,} rows to postcodes_cleaned.csv')
print(f'Columns: {list(df_export.columns)}')
df_export.head()